# Generating an LD Reference Panel

## Description

This notebook generates a linkage-disequilibrium (LD) reference panel from per-chromosome genotype VCFs. For each LD block it extracts variant dosages, mean-imputes missing values, and computes the upper-triangular Pearson correlation matrix between variants, writing one `.cor.xz` matrix and one `.cor.xz.bim` variant list per block. It then builds a per-chromosome `ld_meta_file` table indexing the blocks.

The original protocol panel is generated from ADSP GCAD non-Hispanic white samples; dosages are computed with `cyvcf2` applying a minor-allele-frequency threshold of 0.05%, a minor-allele-count threshold of 5, and a 5% missingness threshold.

> **Synthetic-data note.** The real ADSP genotypes are individual-level, access-controlled human data, so they are **not** used here. This user-friendly version runs end-to-end on a small **synthetic** chr22 genotype VCF (`protocol_example.ld_genotype.chr22.vcf.gz`, 20 samples, clearly labeled `##source=SYNTHETIC...`) purely to demonstrate the pipeline. The resulting correlations are not a real LD reference. To build a real panel, replace the input VCF with genuine genotypes (same format) and adjust the block list.

## Input

- `--genotype-vcf`: a per-chromosome, **bgzipped and tabix-indexed** genotype VCF (`.vcf.gz` + `.vcf.gz.tbi`). Here: `input/ld/protocol_example.ld_genotype.chr22.vcf.gz` (synthetic).
- `--ld-blocks`: an LD-block definition BED file (`chrom`, `start`, `end`): `input/ld_reference/protocol_example.LD_blocks.bed`.
- `--chrom`: the chromosome label used in the VCF and block file (default `chr22`).
- `--maf-min` / `--mac-min` / `--msng-min`: minor-allele-frequency, minor-allele-count and missingness filters (defaults `0.0005` / `5` / `0.05`).
- `--cwd`: output directory (default `output/ld_reference`).

## Output

- One `.cor.xz` file per LD block containing the (upper-triangular) Pearson product-moment correlation coefficients between variants.
- One `.cor.xz.bim` file per block listing the variants in that `.cor.xz`.
- A per-chromosome `ld_meta_file_chr<N>.tsv` table with columns `chrom`, `start`, `end`, `path` indexing all blocks.

## Steps

**Consolidated workflow.** The original recipe has been consolidated into a single parameterized SoS workflow (`default`). It reads the genotype VCF and the LD-block BED, and for each block computes the upper-triangular Pearson correlation matrix (with MAF / MAC / missingness filtering and mean-imputation of missing dosages), writing one `.cor.xz` matrix and one `.cor.xz.bim` variant list per block, then builds the per-chromosome `ld_meta_file`. Run it with:

**Timing:** ~10-30 min on typical compute infrastructure.

In [ ]:
sos run pipeline/ld_reference_generation.ipynb default \
    --genotype-vcf input/ld/protocol_example.ld_genotype.chr22.vcf.gz \
    --ld-blocks input/ld_reference/protocol_example.LD_blocks.bed \
    --chrom chr22 \
    --cwd output/ld_reference

## Command interface

In [ ]:
sos run pipeline/ld_reference_generation.ipynb -h

## Workflow implementation

The consolidated `default` workflow below declares its parameters in `[global]` and does all the work in a single `[default]` section: a Python action computes the per-block correlation matrices (MAF / MAC / missingness filtering + mean-imputation) and writes the `.cor.xz` / `.cor.xz.bim` files, and an R action builds the `ld_meta_file`. The `task:` / `container` settings are kept for parity with the cluster protocol but are not needed for the toy run.

In [ ]:
[global]
# Genotype VCF (bgzipped + tabix-indexed) for one chromosome
parameter: genotype_vcf = path
# LD-block definition BED file (chrom, start, end)
parameter: ld_blocks = path
# Chromosome label used in the VCF / block file
parameter: chrom = "chr22"
# Output directory
parameter: cwd = path("output/ld_reference")
# Minor-allele-frequency threshold
parameter: maf_min = 0.0005
# Minor-allele-count threshold
parameter: mac_min = 5
# Missingness-rate threshold
parameter: msng_min = 0.05
# Container (kept for parity with the protocol; unused in the MWE)
parameter: container = ""
# Resource options (kept for parity with the protocol)
parameter: job_size = 1
parameter: walltime = "5h"
parameter: mem = "8G"
parameter: numThreads = 1

In [ ]:
[default]
# One LD reference panel per chromosome: per-block .cor.xz + .cor.xz.bim, plus the chromosome ld_meta_file
input: genotype_vcf, ld_blocks
output: f'{cwd}/ld_meta_file_{chrom}.tsv'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output:bn}'
python: expand = "${ }", stderr = f'{_output:nn}.stderr', stdout = f'{_output:nn}.stdout'
    import os
    import numpy as np
    from math import nan
    from cyvcf2 import VCF
    import lzma

    chrom = "${chrom}"
    maf_min, mac_min, msng_min = ${maf_min}, ${mac_min}, ${msng_min}
    vcf_files = {chrom: ["${genotype_vcf}"]}

    def fill_missing_with_row_means(data):
        row_means = np.nanmean(data, axis=1)
        inds = np.where(np.isnan(data))
        data[inds] = np.take(row_means, inds[0])
        return data

    def get_dosages(vcf_obj, maf_min=0.0005, mac_min=5, msng_min=0.05):
        arr = []; var_names = []
        for var in vcf_obj:
            if len(var.ALT) != 1:
                continue
            dosage = [sum(x[0:2]) for x in [[nan if x1 == -1 else x1 for x1 in x0] for x0 in var.genotypes]]
            if np.nanvar(dosage) == 0:
                continue
            counts = [np.nansum([2 - x for x in dosage]), np.nansum(dosage)]
            nan_count = np.sum(np.isnan(dosage))
            num_samp_non_na = len(dosage) - nan_count
            mac = min(counts)
            maf = mac / num_samp_non_na
            msng_rate = nan_count / (num_samp_non_na + nan_count)
            if (maf < maf_min) or (mac < mac_min) or (msng_rate > msng_min):
                continue
            arr.append(dosage)
            var_names.append(var.CHROM + ":" + "_".join([str(var.POS), var.REF, var.ALT[0]]))
        if len(var_names) != 0:
            return fill_missing_with_row_means(np.array(arr)), var_names
        else:
            return np.array([[]]), var_names

    def get_dosages_range(vcf_obj, chrm, start, end):
        vcf_qry_str = chrm + ":" + str(start) + "-" + str(end)
        return get_dosages(vcf_obj(vcf_qry_str), maf_min, mac_min, msng_min)

    def flatten(xss):
        return [x for xs in xss for x in xs]

    ld_blocks = []
    with open("${ld_blocks}") as f:
        for line in f:
            elems = line.split()
            elems[-1] = elems[-1].strip()
            ld_blocks.append(elems)

    os.makedirs("${cwd}", exist_ok=True)
    for block in ld_blocks:
        chrm, start, end = block
        out = "${cwd}/%s_%s_%s.cor.xz" % (chrm, start, end)
        dosages = [get_dosages_range(VCF(v), chrm, start, end) for v in vcf_files[chrm]]
        variants = flatten([x[1] for x in dosages])
        dosage = np.concatenate([x[0] for x in dosages if len(x[1]) > 0], axis=0)
        cor = np.triu(np.corrcoef(dosage))
        with open(out + ".bim", "w+") as f:
            for var in variants:
                cc = var.split(":")[0]
                pos, ref, alt = var.replace(cc + ":", "").split("_")
                elems = [cc.replace("chr", ""), var, "0", pos, alt.strip(), ref]
                f.write("\t".join(elems)); f.write("\n")
        with lzma.open(out, "w", preset=9) as f:
            for r in range(cor.shape[0]):
                f.write((" ".join(["{:.6f}".format(x) for x in cor[r, :]])).encode()); f.write(b"\n")
        print("wrote", out, "(", len(variants), "variants )")

R: expand = "${ }", stderr = f'{_output:nn}.R.stderr', stdout = f'{_output:nn}.R.stdout'
    library(dplyr)
    library(stringr)
    generate_ld_meta_file = function(ld_meta_file_path){
        ld_meta_file <- list.files(ld_meta_file_path, pattern = "\\.bim$") %>%
        data.frame(path = .) %>%
        mutate(
            chrom = str_extract(path, "^[^_]+"),
            start_end = str_extract(path, "_[0-9]+_[0-9]+"),
            start = as.numeric(str_extract(start_end, "[0-9]+")),
            end = as.numeric(str_extract(start_end, "[0-9]+$")),
            path = paste0(basename(ld_meta_file_path), "/",chrom, "_", start, "_", end, ".cor.xz",
                          ",", basename(ld_meta_file_path), "/",chrom, "_", start, "_", end, ".cor.xz.bim")
        )%>%select(chrom,start,end,path)
        return(ld_meta_file)
    }
    ld_meta <- generate_ld_meta_file("${cwd}")
    write.table(ld_meta, "${_output}", sep="\t", col.names=TRUE, row.names=FALSE, quote=FALSE)
    print(ld_meta)

## Troubleshooting

| Step | Problem | Possible Reason | Solution |
|------|---------|-----------------|----------|
| 5 | Region query returns no variants | VCF not tabix-indexed, or `chrom` naming mismatch (`chr22` vs `22`) | Ensure the VCF is bgzipped + `tabix`-indexed and the block `chrom` matches the VCF `CHROM` field |
| 5 | `ModuleNotFoundError: xz` | The `xz` PyPI package is not installed | Use the standard-library `lzma` module (as done here); both write the `.xz` format |

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.